# Model 4 · TF-IDF + Deepnet

Same MLP head as Model 3, but over SVD-reduced TF-IDF features instead of
MPNet embeddings — isolates how much of Model 3's performance comes from the
pre-trained encoder vs. the head architecture. Includes Model 1's cells as a
prerequisite (needed for the fitted `tfidf` vectorizer).


In [ ]:
import os, re, gc, math, random, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Config - CPU only
# ---------------------------------------------------------------------------
BASE        = "../data"
TRAIN_PATH  = f"{BASE}/train.csv"
TEST_PATH   = f"{BASE}/test.csv"
OUTPUT_PATH = "../outputs/submission.csv"

OPTIONS  = ["A", "B", "C", "D", "E"]
SEED     = 42
VAL_SIZE = 0.20
N_FOLDS  = 5

MPNET_ID = "sentence-transformers/all-mpnet-base-v2"

# ---------------------------------------------------------------------------
# Weights & Biases - one run per model, so every model has a tracked run with
# comparable metrics (MAP@3, accuracy, macro F1, weighted F1).
# Wrapped so that a W&B failure can never abort the run or lose the submission.
# ---------------------------------------------------------------------------
WANDB_PROJECT = "23f2004742-t22026"
WANDB_ON = True

os.environ["WANDB_SILENT"] = "true"

try:
    import wandb
    _key = None
    try:
        from kaggle_secrets import UserSecretsClient
        _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        _key = os.environ.get("WANDB_API_KEY")

    if _key:
        wandb.login(key=_key)
        print(f"W&B ready -> project '{WANDB_PROJECT}'")
    else:
        # A bare wandb.login() waits on stdin, which would hang a
        # "Save & Run All" commit forever. Disable instead of risking that.
        WANDB_ON = False
        print("W&B key not found (add WANDB_API_KEY as a Kaggle secret).")
        print("Continuing without tracking; all metrics are still printed.")
except Exception as e:
    WANDB_ON = False
    print(f"W&B unavailable ({type(e).__name__}); metrics still printed locally")

# DeBERTa is kept as an experiment only. It is a 435M model: fine-tuning it on
# CPU is not practical, so it stays off unless a GPU is attached.
RUN_DEBERTA = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print(f"torch {torch.__version__} | device: {DEVICE}")
print(f"DeBERTa experiment: {'ON' if RUN_DEBERTA else 'OFF (no GPU - expected on CPU)'}")


## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")

counts = train_df["answer"].value_counts().reindex(OPTIONS)
probs  = counts / counts.sum()
order  = list(probs.sort_values(ascending=False).index)

PRIOR_MAP3 = probs[order[0]] + probs[order[1]] / 2 + probs[order[2]] / 3
LABEL_LOGPRIOR = np.log(probs[OPTIONS].values.astype(np.float64))

print("\nAnswer distribution:")
for o in order:
    print(f"  {o}  {counts[o]:>4}  ({probs[o]:.1%})")
print(f"\nRandom-ordering MAP@3        : 0.3667")
print(f"Always '{' '.join(order[:3])}' MAP@3          : {PRIOR_MAP3:.4f}   <- the bar to beat")

train_df.head(3)


## 2. Preprocessing and split

Prompts carry boilerplate prefixes ("Pick the best possible answer:", …) which
are stripped. **The split is made on the CLEANED prompt**, because two prompts
that differ only by prefix become identical after cleaning - splitting on raw
text would put the same question on both sides and make validation meaningless.


In [ ]:
BOILERPLATE = [
    r"^Pick the best possible answer:\s*", r"^Choose the correct answer:\s*",
    r"^Select the most accurate option:\s*", r"^Identify the correct statement:\s*",
    r"^Determine the correct option:\s*", r"^Which of the following\s*",
    r"\s*among the listed options\.?$", r"\s*from the following choices\.?$",
    r"\s*based on the given context\.?$", r"\s*carefully\.?$",
]


def clean_text(t):
    t = re.sub(r"\s+", " ", str(t)).strip()
    for p in BOILERPLATE:
        t = re.sub(p, "", t, flags=re.IGNORECASE).strip()
    return t


def clean_frame(df):
    out = df.copy()
    for c in ["prompt"] + OPTIONS:
        out[c] = out[c].map(clean_text)
    return out


train = clean_frame(train_df)
test  = clean_frame(test_df)

n_raw, n_clean = train_df["prompt"].nunique(), train["prompt"].nunique()
print(f"Unique prompts  raw {n_raw}  ->  cleaned {n_clean}   "
      f"({n_raw - n_clean} collapsed by preprocessing)")

uniq = train["prompt"].unique()
tr_p, va_p = train_test_split(uniq, test_size=VAL_SIZE, random_state=SEED)

train_split = train[train["prompt"].isin(tr_p)].reset_index(drop=True)
val_split   = train[train["prompt"].isin(va_p)].reset_index(drop=True)
assert not (set(train_split["prompt"]) & set(val_split["prompt"]))

y_val   = val_split["answer"].tolist()
y_train = train["answer"].tolist()

print(f"Train {len(train_split)} | Val {len(val_split)} | no cleaned-prompt overlap")


## 3. Metrics - MAP@3, Accuracy, Macro F1

In [ ]:
def average_precision_at_3(actual, predicted):
    for rank, p in enumerate(predicted[:3], start=1):
        if p == actual:
            return 1.0 / rank
    return 0.0


def map_at_3(actuals, preds):
    return float(np.mean([average_precision_at_3(a, p) for a, p in zip(actuals, preds)]))


def top3(scores):
    """(n,5) score matrix -> list of top-3 letter lists."""
    return [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in np.asarray(scores)]


def wandb_run(name, metrics, config=None):
    """One W&B run per model. Never allowed to break the pipeline."""
    if not WANDB_ON:
        return
    try:
        wandb.init(project=WANDB_PROJECT, name=name, reinit=True,
                   config=config or {})
        wandb.log({k.lower().replace("@", "").replace("+", "_"): float(v)
                   for k, v in metrics.items()})
        wandb.finish()
    except Exception as e:
        print(f"    (W&B log skipped: {type(e).__name__})")


def evaluate(name, actuals, scores, store=None, log=True, config=None):
    preds = top3(scores)
    t1 = [p[0] for p in preds]
    m = {
        "MAP@3":    map_at_3(actuals, preds),
        "Accuracy": accuracy_score(actuals, t1),
        "MacroF1":  f1_score(actuals, t1, labels=OPTIONS, average="macro", zero_division=0),
        "WgtF1":    f1_score(actuals, t1, labels=OPTIONS, average="weighted", zero_division=0),
    }
    print(f"{name:<28} MAP@3 {m['MAP@3']:.4f} | Acc {m['Accuracy']:.4f} | "
          f"MacroF1 {m['MacroF1']:.4f} | WgtF1 {m['WgtF1']:.4f}")
    if store is not None:
        store[name] = m
    if log:
        wandb_run(name.strip().replace(" ", "-").lower(), m, config)
    return m


RESULTS = {}
print(f"reference: random 0.3667 | prior {PRIOR_MAP3:.4f}")


In [ ]:
# Row positions of the validation split within the cleaned training frame.
# (Hoisted from the source notebook's Model 2 cell -- every model below needs it.)
val_pos = train.index[train["prompt"].isin(va_p)].to_numpy()


## Model 1 - TF-IDF + cosine similarity  *(baseline)*

Each option is scored by the TF-IDF cosine similarity between the **question**
and the **option text**. Purely lexical: it rewards word overlap, so it fails
whenever the correct answer paraphrases the question instead of repeating it.


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                        strip_accents="unicode", min_df=1)
# fit on all text (no labels used) so the test vocabulary is represented
tfidf.fit(pd.concat([train["prompt"], test["prompt"]] +
                    [train[o] for o in OPTIONS] + [test[o] for o in OPTIONS]).astype(str))


def tfidf_scores(df):
    q = tfidf.transform(df["prompt"].astype(str))
    out = np.zeros((len(df), len(OPTIONS)), dtype=np.float32)
    for j, o in enumerate(OPTIONS):
        opt = tfidf.transform(df[o].astype(str))
        out[:, j] = np.asarray(q.multiply(opt).sum(axis=1)).ravel() / (
            (np.sqrt(q.multiply(q).sum(axis=1)).A.ravel() *
             np.sqrt(opt.multiply(opt).sum(axis=1)).A.ravel()) + 1e-9)
    return out


tfidf_val   = tfidf_scores(val_split)
tfidf_train = tfidf_scores(train)
tfidf_test  = tfidf_scores(test)

evaluate("1. TF-IDF cosine", y_val, tfidf_val, RESULTS)


## Model 4 - TF-IDF + Deepnet  *(supervised)*

The same head, but over **TF-IDF** features instead of MPNet embeddings. The
sparse vectors are reduced with truncated SVD so the dense head can consume them.
This isolates how much of Model 3's performance comes from the neural head versus
from MPNet's semantics.


In [ ]:
from sklearn.decomposition import TruncatedSVD

SVD_DIM = 256
print(f"Reducing TF-IDF to {SVD_DIM} dims...")

all_q = tfidf.transform(pd.concat([train["prompt"], test["prompt"]]).astype(str))
all_o = tfidf.transform(
    [str(r[o]) for _, r in train.iterrows() for o in OPTIONS] +
    [str(r[o]) for _, r in test.iterrows() for o in OPTIONS])

svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
svd.fit(all_q)

def svd_q(df):
    return normalize(svd.transform(tfidf.transform(df["prompt"].astype(str)))).astype(np.float32)

def svd_o(df):
    m = svd.transform(tfidf.transform([str(r[o]) for _, r in df.iterrows() for o in OPTIONS]))
    return normalize(m).reshape(len(df), len(OPTIONS), -1).astype(np.float32)

Qt_train, Ot_train = svd_q(train), svd_o(train)
Qt_test,  Ot_test  = svd_q(test),  svd_o(test)
print(f"explained variance: {svd.explained_variance_ratio_.sum():.1%}")

print("Training TF-IDF + Deepnet, 5-fold...")
tdeep_oof, tdeep_test = run_folds(
    fit_fn=lambda idx, yy, sd: fit_deepnet(Qt_train[idx], Ot_train[idx], yy, seed=sd),
    pred_fn=lambda m, idx, which: deepnet_predict(
        m, Qt_train[idx] if which == "train" else Qt_test,
        Ot_train[idx] if which == "train" else Ot_test),
    n_rows=len(train), n_test=len(test), tag="tfidf-deep", n_seeds=3)

tdeep_val = tdeep_oof[val_pos]
evaluate("4. TF-IDF + Deepnet", y_val, tdeep_val, RESULTS)
